# Enrollment vs assigned billing
### Inscripciones vs conceptos asignados

**Executed public case study · completely synthetic · period 2030-09**

Academic enrollment and billing preparation are separate operational universes. Missing assignments can leave activity outside billing; extra assignments can indicate an incorrect scope. A grand total cannot show which student/program needs review.

Las inscripciones académicas y la preparación de facturación pertenecen a universos operativos distintos. Un faltante puede dejar actividad fuera del circuito de cobro; un exceso puede indicar un alcance incorrecto. El total general no identifica qué alumno/carrera requiere revisión.

[Case study / caso](../docs/cases/01_enrollment.md) · [Español](../docs/cases/01_enrollment.es.md)

## 1. Universe and contracts / Universo y contratos

enrollments.csv (academic events), assigned.csv (assigned lines), cash.csv (synthetic explanatory events). All rows have a unique event ID and explicit period.

The comparison grain is **student + program after selecting period**. Every event must have a unique row ID. Currency values are fictional. No Oracle connection or environment secret is read.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "pyproject.toml").exists() and (p / "data" / "synthetic").is_dir())
sys.path.insert(0, str(root / "src"))
from financial_controls.demo import load_inputs
from financial_controls.contracts import scope, unique, KEYS
from financial_controls.money import money_column, format_cents
from financial_controls.reconciliation import reconcile, summarize
from financial_controls.controls import enrollment_control, billing_control, scholarship_control, units
from financial_controls.reporting import summarize_review, display_detail

period = "2030-09"
data = load_inputs(root)

In [2]:
manifest = pd.DataFrame([
    {"source": name, "rows": len(data[name]), "columns": len(data[name].columns)}
    for name in ["enrollments","assigned","cash"]
])
display(manifest)
print("enrollments")
display(data["enrollments"])

print("assigned")
display(data["assigned"])

print("cash")
display(data["cash"])

,source,rows,columns
0,enrollments,10,5
1,assigned,9,6
2,cash,1,4


enrollments


,row_id,student_ref,program_ref,period,chargeable
0,SYN-E01,SYN-S001,SYN-A,2030-09,Y
1,SYN-E02,SYN-S001,SYN-A,2030-09,Y
2,SYN-E03,SYN-S001,SYN-B,2030-09,Y
3,SYN-E04,SYN-S002,SYN-A,2030-09,Y
4,SYN-E05,SYN-S002,SYN-A,2030-09,N
5,SYN-E06,SYN-S003,SYN-B,2030-09,Y
6,SYN-E07,SYN-S003,SYN-B,2030-09,Y
7,SYN-E08,SYN-S004,SYN-A,2030-09,Y
8,SYN-E09,SYN-S005,SYN-A,2030-09,Y
9,SYN-E10,SYN-S007,SYN-A,2030-09,Y


assigned


,row_id,student_ref,program_ref,period,amount,prior_period
0,SYN-A01,SYN-S001,SYN-A,2030-09,100.00,N
1,SYN-A02,SYN-S001,SYN-A,2030-09,100.00,N
2,SYN-A03,SYN-S001,SYN-B,2030-09,50.00,N
3,SYN-A04,SYN-S002,SYN-A,2030-09,100.00,N
4,SYN-A05,SYN-S003,SYN-B,2030-09,150.00,N
5,SYN-A06,SYN-S003,SYN-B,2030-09,150.00,N
6,SYN-A07,SYN-S004,SYN-A,2030-09,100.00,N
7,SYN-A08,SYN-S004,SYN-A,2030-09,100.00,Y
8,SYN-A09,SYN-S006,SYN-B,2030-09,80.00,N


cash


,row_id,student_ref,program_ref,period
0,SYN-C01,SYN-S007,SYN-A,2030-09


## 2. Normalize and aggregate / Normalizar y agregar

Keys stay strings; null keys are not converted to zero. Money is parsed from decimal strings and becomes integer cents before summing. Cardinality is declared after grouping.

In [3]:
enrollments = scope(data["enrollments"], period, ["chargeable"])
assigned = scope(data["assigned"], period, ["prior_period"])
expected = summarize(units(enrollments))
actual = summarize(units(assigned))
display(expected.rename(columns={"lines": "enrollment_lines"}))
display(actual.rename(columns={"lines": "assigned_lines"}))
preview = expected.merge(actual, on=KEYS, how="outer", validate="one_to_one",
                         suffixes=("_expected", "_actual"), indicator=True)
display(preview)

,student_ref,program_ref,enrollment_lines,cents
0,SYN-S001,SYN-A,2,0
1,SYN-S001,SYN-B,1,0
2,SYN-S002,SYN-A,2,0
3,SYN-S003,SYN-B,2,0
4,SYN-S004,SYN-A,1,0
5,SYN-S005,SYN-A,1,0
6,SYN-S007,SYN-A,1,0


,student_ref,program_ref,assigned_lines,cents
0,SYN-S001,SYN-A,2,0
1,SYN-S001,SYN-B,1,0
2,SYN-S002,SYN-A,1,0
3,SYN-S003,SYN-B,2,0
4,SYN-S004,SYN-A,2,0
5,SYN-S006,SYN-B,1,0


,student_ref,program_ref,lines_expected,cents_expected,lines_actual,cents_actual,_merge
0,SYN-S001,SYN-A,2.0,0.0,2.0,0.0,both
1,SYN-S001,SYN-B,1.0,0.0,1.0,0.0,both
2,SYN-S002,SYN-A,2.0,0.0,1.0,0.0,both
3,SYN-S003,SYN-B,2.0,0.0,2.0,0.0,both
4,SYN-S004,SYN-A,1.0,0.0,2.0,0.0,both
5,SYN-S005,SYN-A,1.0,0.0,NaN,NaN,left_only
6,SYN-S006,SYN-B,NaN,NaN,1.0,0.0,right_only
7,SYN-S007,SYN-A,1.0,0.0,NaN,NaN,left_only


## 3. A contract must fail when violated / Un contrato debe fallar

A duplicate event or rule must not multiply rows silently. The following deliberate synthetic error is caught and reported by category only.

In [4]:
bad_input = pd.concat([data["assigned"], data["assigned"].iloc[[0]]], ignore_index=True)
try:
    scope(bad_input, period, ["amount"])
except ValueError as error:
    print(type(error).__name__ + ": " + str(error))
else:
    raise AssertionError("Duplicate event should fail")

ValueError: Duplicate key violates cardinality contract


## 4. Compare, classify and explain / Comparar, clasificar y justificar

Cash and explicitly non-chargeable rows can explain missing assignments (negative direction). Assigned rows marked prior_period can explain extra assignments (positive direction). Capacity never exceeds the residual. Null chargeability is rejected rather than interpreted as a free subject.

Contado y filas explícitamente no aranceladas pueden explicar asignaciones faltantes (dirección negativa). Las asignaciones marcadas prior_period pueden explicar excesos (dirección positiva). La capacidad nunca supera el remanente. Un indicador arancelario nulo se rechaza.

Difference = actual − expected. Raw status and review status are separate.

In [5]:
result = enrollment_control(data["enrollments"], data["assigned"], data["cash"], period=period)
display(display_detail(result))
display(summarize_review(result))

,student_ref,program_ref,lines_expected,lines_actual,amount_expected,amount_actual,difference_lines,difference_amount,justified_amount,pending_amount,raw_status,review_status,presence
0,SYN-S001,SYN-A,2,2,0.00,0.00,0,0.00,0.00,0.00,OK,OK,both
1,SYN-S001,SYN-B,1,1,0.00,0.00,0,0.00,0.00,0.00,OK,OK,both
2,SYN-S002,SYN-A,2,1,0.00,0.00,-1,0.00,0.00,0.00,DE MENOS,JUSTIFICADO,both
3,SYN-S003,SYN-B,2,2,0.00,0.00,0,0.00,0.00,0.00,OK,OK,both
4,SYN-S004,SYN-A,1,2,0.00,0.00,1,0.00,0.00,0.00,DE MÁS,JUSTIFICADO,both
5,SYN-S005,SYN-A,1,0,0.00,0.00,-1,0.00,0.00,0.00,DE MENOS,PENDIENTE,left_only
6,SYN-S006,SYN-B,0,1,0.00,0.00,1,0.00,0.00,0.00,DE MÁS,PENDIENTE,right_only
7,SYN-S007,SYN-A,1,0,0.00,0.00,-1,0.00,0.00,0.00,DE MENOS,JUSTIFICADO,left_only


,raw_status,review_status,cases,students,difference_lines,difference_cents,justified_cents,pending_cents
0,DE MENOS,JUSTIFICADO,2,2,-2,0,0,0
1,DE MENOS,PENDIENTE,1,1,-1,0,0,0
2,DE MÁS,JUSTIFICADO,1,1,1,0,0,0
3,DE MÁS,PENDIENTE,1,1,1,0,0,0
4,OK,OK,3,2,0,0,0,0


## 5. Evidence is consumed, not assumed / Evidencia consumida, no supuesta

The allocation ledger preserves the evidence ID, rule, direction and consumed capacities. Original difference = signed justification + pending. A partial explanation leaves the case open.

In [6]:
display(result.allocations)
for measure in ["lines", "cents"]:
    assert (result.detail[f"difference_{measure}"] ==
            result.detail[f"justified_{measure}"] + result.detail[f"pending_{measure}"]).all()
display(result.detail.loc[result.detail["review_status"].eq("PENDIENTE"),
                          [*KEYS, "pending_lines", "pending_cents", "raw_status"]])

,student_ref,program_ref,evidence_id,rule,direction,lines,cents
0,SYN-S007,SYN-A,SYNTHETIC_CASH:SYN-C01,SYNTHETIC_CASH,-1,1,0
1,SYN-S002,SYN-A,SYNTHETIC_NON_CHARGEABLE:SYN-E05,SYNTHETIC_NON_CHARGEABLE,-1,1,0
2,SYN-S004,SYN-A,SYNTHETIC_PRIOR_PERIOD:SYN-A08,SYNTHETIC_PRIOR_PERIOD,1,1,0


,student_ref,program_ref,pending_lines,pending_cents,raw_status
5,SYN-S005,SYN-A,-1,0,DE MENOS
6,SYN-S006,SYN-B,1,0,DE MÁS


## 6. A failure totals alone would miss / Lo que el total oculta

9 assigned − 10 enrollment = −1 net line = sum of keyed differences. After explanations, pending lines net to zero but their absolute sum is 2. Two unresolved keys therefore remain despite a zero pending net.

9 asignadas − 10 inscriptas = −1 línea neta = suma de diferencias por clave. Tras justificar, las líneas pendientes netean cero, pero suman 2 en valor absoluto. Por eso siguen existiendo dos claves pendientes.

In [7]:
assert result.kpis["pending_net_lines"] == 0
assert result.kpis["pending_absolute_lines"] == 2
assert result.kpis["pending_cases"] == 2
display(result.detail.loc[result.detail["student_ref"].eq("SYN-S001"),
                          [*KEYS, "lines_expected", "lines_actual"]])

,student_ref,program_ref,lines_expected,lines_actual
0,SYN-S001,SYN-A,2,2
1,SYN-S001,SYN-B,1,1


## 7. Independent source-to-detail bridge / Puente independiente

Totals are calculated before joining. Unassigned amounts remain separate from keyed pending exposure. A complete bridge is not a declaration that all business exceptions are resolved.

In [8]:
bridge = pd.DataFrame([
    {"measure": m, "expected": result.kpis[f"expected_{m}"],
     "actual": result.kpis[f"actual_{m}"],
     "mother_difference": result.kpis[f"mother_difference_{m}"],
     "keyed_detail": result.kpis[f"detail_difference_{m}"],
     "unassigned": result.kpis[f"unassigned_{m}"]}
    for m in ["lines", "cents"]
])
display(bridge)
assert (bridge["mother_difference"] == bridge["keyed_detail"] + bridge["unassigned"]).all()
display(pd.DataFrame([{
    "pending_cases": result.kpis["pending_cases"],
    "pending_net_cents": result.kpis["pending_net_cents"],
    "pending_absolute_cents": result.kpis["pending_absolute_cents"],
    "unassigned_cents": result.kpis["unassigned_cents"],
    "control_clear": result.kpis["control_clear"]
}]))

,measure,expected,actual,mother_difference,keyed_detail,unassigned
0,lines,10,9,-1,-1,0
1,cents,0,0,0,0,0


,pending_cases,pending_net_cents,pending_absolute_cents,unassigned_cents,control_clear
0,2,0,0,0,False


## 8. Pipeline trace / Trazabilidad por etapas

The stage ledger records counts and check categories, never connection data or source records. Run the CLI with --verbose to see the same metadata in logging.

In [9]:
display(pd.DataFrame(result.trace))
assert result.trace[-1]["stage"] == "totals_bridge"
assert result.trace[-1]["check"] == "PASS"

,stage,rows,check
0,scope.enrollments,10,validated_keys_and_row_ids; excluded_rows=0
1,scope.assigned,9,validated_keys_and_row_ids; excluded_rows=0
2,scope.cash,1,validated_keys_and_row_ids; excluded_rows=0
3,input_contract,19,integer_measures;critical_keys;nonnegative_counts
4,aggregate.expected,7,unique_student_program
5,aggregate.actual,6,unique_student_program
6,join.outer,8,one_to_one;both_universes_preserved
7,classify,8,original_differences_preserved
8,allocate_evidence,3,unique_evidence;capacity_capped;direction_checked
9,totals_bridge,8,PASS


## Review outputs and limits / Outputs y límites

Run `python -m financial_controls.demo --period 2030-09 --output artifacts` to export detail, summary, allocations, unassigned records, trace, KPIs and HTML.

Equal counts can hide different subjects inside the same key. Aggregate evidence does not prove which subject was covered. A prior-period flag is a synthetic contract, not a production date-detection algorithm.

Igual cantidad puede ocultar materias distintas dentro de la clave. La evidencia agregada no prueba qué materia quedó cubierta. prior_period es un contrato ficticio, no un algoritmo productivo de detección de fechas.

### Interview / Entrevista
- Why is a composite key safer than a student-only total? / ¿Por qué usar clave compuesta en lugar del total por alumno?
- Why does a zero pending net still require review? / ¿Por qué un neto pendiente cero requiere revisión?
- When should this become exact subject-level matching? / ¿Cuándo corresponde pasar a un matching exacto por materia?

No production processing, operational impact metric or confidential rule is reproduced.